In [ ]:
"""
MODULE 2: EXPLORATORY DATA ANALYSIS & PREPROCESSING
================================================================================

Research Context:
This module conducts comprehensive exploratory analysis of socioeconomic
deprivation patterns across Karnataka districts.

Objectives:
1. Understand distributional properties of proxy indicators
2. Identify correlations and multicollinearity
3. Detect dominant deprivation dimensions
4. Prepare features for machine learning analysis

Statistical Rigor:
- Descriptive statistics with outlier detection
- Correlation analysis with significance testing
- Dimension reduction exploration
- Feature engineering for ML readiness

Author: Senior Data Scientist
Purpose: Publication-quality exploratory analysis
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path

# Set professional plotting defaults
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.1)
sns.set_palette("husl")

print("="*80)
print("MODULE 2: EXPLORATORY DATA ANALYSIS & PREPROCESSING")
print("="*80)


from google.colab import drive
drive.mount("/content/drive", force_remount=True)



BASE_DIR = "/content/drive/MyDrive/deprivation_analysis"
INPUT_DATA = f"{BASE_DIR}/output/module1_processed_data.csv"
OUTPUT_DIR = f"{BASE_DIR}/output"

# ============================================================================
# SECTION 2.1: DATA LOADING
# ============================================================================

def load_module1_output(filepath):
    """
    Load processed data from Module 1.

    Parameters:
    -----------
    filepath : str
        Path to Module 1 output CSV

    Returns:
    --------
    pd.DataFrame : Processed district data with SEDI scores
    """

    print("\n[2.1] Loading Module 1 Output...")

    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"Module 1 output not found: {filepath}\n"
            "Please run Module 1 first to generate processed data."
        )

    df = pd.read_csv(filepath)
    print(f"✓ Data loaded: {df.shape[0]} districts, {df.shape[1]} variables")

    return df


# ============================================================================
# SECTION 2.2: DESCRIPTIVE STATISTICS
# ============================================================================

def descriptive_statistics(df, output_dir='output'):
    """
    Compute comprehensive descriptive statistics for all indicators.

    Metrics computed:
    - Central tendency: Mean, Median
    - Dispersion: Std Dev, IQR, Range
    - Shape: Skewness, Kurtosis
    - Outlier detection: Z-scores, IQR method

    Parameters:
    -----------
    df : pd.DataFrame
        District data
    output_dir : str
        Directory for output files

    Returns:
    --------
    pd.DataFrame : Summary statistics table
    """

    print("\n[2.2] Computing Descriptive Statistics...")

    # Select raw (non-normalized) socioeconomic indicators
    indicator_cols = [
        'Per_Capita_Income',
        'Literacy_Rate',
        'Healthcare_Facilities_Per_Lakh',
        'Road_Density',
        'Electricity_Access',
        'Unemployment_Rate',
        'Urbanization_Rate'
    ]

    # Filter to available columns
    indicator_cols = [col for col in indicator_cols if col in df.columns]

    # Compute statistics
    stats_dict = {}

    for col in indicator_cols:
        stats_dict[col] = {
            'Mean': df[col].mean(),
            'Median': df[col].median(),
            'Std Dev': df[col].std(),
            'Min': df[col].min(),
            'Max': df[col].max(),
            'Range': df[col].max() - df[col].min(),
            'Q1': df[col].quantile(0.25),
            'Q3': df[col].quantile(0.75),
            'IQR': df[col].quantile(0.75) - df[col].quantile(0.25),
            'Skewness': df[col].skew(),
            'Kurtosis': df[col].kurtosis(),
            'CV': (df[col].std() / df[col].mean()) * 100  # Coefficient of Variation
        }

    # Convert to DataFrame
    stats_df = pd.DataFrame(stats_dict).T

    print("✓ Descriptive statistics computed")
    print("\nKey Findings:")
    print(f"  • Most variable indicator (highest CV): {stats_df['CV'].idxmax()}")
    print(f"  • Most skewed indicator: {stats_df['Skewness'].abs().idxmax()}")

    # Save to CSV
    os.makedirs(output_dir, exist_ok=True)
    stats_path = os.path.join(output_dir, 'module2_descriptive_stats.csv')
    stats_df.to_csv(stats_path)
    print(f"✓ Saved: {stats_path}")

    return stats_df


def detect_outliers(df, method='iqr', threshold=1.5):
    """
    Detect outliers in socioeconomic indicators.

    Methods:
    - 'iqr': Interquartile Range method (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
    - 'zscore': Z-score method (|z| > 3)

    Parameters:
    -----------
    df : pd.DataFrame
        District data
    method : str
        Outlier detection method
    threshold : float
        IQR multiplier (for IQR method) or z-score threshold

    Returns:
    --------
    dict : Outlier detection results by indicator
    """

    print(f"\n[2.3] Detecting Outliers (Method: {method})...")

    indicator_cols = [
        'Per_Capita_Income', 'Literacy_Rate', 'Healthcare_Facilities_Per_Lakh',
        'Road_Density', 'Electricity_Access', 'Unemployment_Rate', 'Urbanization_Rate'
    ]
    indicator_cols = [col for col in indicator_cols if col in df.columns]

    outlier_results = {}

    for col in indicator_cols:
        if method == 'iqr':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR

            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

        elif method == 'zscore':
            z_scores = np.abs(stats.zscore(df[col]))
            outliers = df[z_scores > threshold]

        outlier_results[col] = {
            'n_outliers': len(outliers),
            'outlier_districts': outliers['District'].tolist() if len(outliers) > 0 else [],
            'outlier_values': outliers[col].tolist() if len(outliers) > 0 else []
        }

        if len(outliers) > 0:
            print(f"  • {col}: {len(outliers)} outliers detected")

    print("✓ Outlier detection complete")

    return outlier_results


# ============================================================================
# SECTION 2.3: CORRELATION ANALYSIS
# ============================================================================

def correlation_analysis(df, output_dir='output'):
    """
    Perform comprehensive correlation analysis between indicators.

    Analyses:
    1. Pearson correlation (linear relationships)
    2. Spearman correlation (monotonic relationships)
    3. Statistical significance testing
    4. Multicollinearity detection (VIF)

    Parameters:
    -----------
    df : pd.DataFrame
        District data
    output_dir : str
        Output directory for plots

    Returns:
    --------
    tuple : (pearson_corr, spearman_corr, vif_df)
    """

    print("\n[2.4] Correlation Analysis...")

    # Select raw indicators
    indicator_cols = [
        'Per_Capita_Income', 'Literacy_Rate', 'Healthcare_Facilities_Per_Lakh',
        'Road_Density', 'Electricity_Access', 'Unemployment_Rate', 'Urbanization_Rate'
    ]
    indicator_cols = [col for col in indicator_cols if col in df.columns]

    # Compute Pearson correlation
    pearson_corr = df[indicator_cols].corr(method='pearson')

    # Compute Spearman correlation
    spearman_corr = df[indicator_cols].corr(method='spearman')

    print("✓ Correlation matrices computed")

    # Identify strongest correlations
    # Get upper triangle to avoid duplicates
    mask = np.triu(np.ones_like(pearson_corr), k=1).astype(bool)
    upper_tri = pearson_corr.where(mask)

    # Find highest correlations
    high_corr = []
    for col in upper_tri.columns:
        for idx in upper_tri.index:
            val = upper_tri.loc[idx, col]
            if pd.notna(val) and abs(val) > 0.5:  # Moderate to strong correlation
                high_corr.append((idx, col, val))

    if high_corr:
        print("\nStrong Correlations Detected (|r| > 0.5):")
        for var1, var2, corr in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True)[:5]:
            print(f"  • {var1} ↔ {var2}: r = {corr:.3f}")

    # Compute VIF for multicollinearity detection
    vif_df = compute_vif(df[indicator_cols])

    # Create correlation visualizations
    visualize_correlations(pearson_corr, spearman_corr, output_dir)

    # Save correlation matrices
    os.makedirs(output_dir, exist_ok=True)
    pearson_corr.to_csv(os.path.join(output_dir, 'module2_pearson_correlation.csv'))
    spearman_corr.to_csv(os.path.join(output_dir, 'module2_spearman_correlation.csv'))
    print(f"✓ Correlation matrices saved")

    return pearson_corr, spearman_corr, vif_df


def compute_vif(df):
    """
    Compute Variance Inflation Factor to detect multicollinearity.

    VIF Interpretation:
    - VIF = 1: No correlation
    - 1 < VIF < 5: Moderate correlation (acceptable)
    - VIF > 5: High multicollinearity (problematic)
    - VIF > 10: Severe multicollinearity (remove variable)

    Parameters:
    -----------
    df : pd.DataFrame
        Data with numeric features

    Returns:
    --------
    pd.DataFrame : VIF scores for each variable
    """

    from statsmodels.stats.outliers_influence import variance_inflation_factor

    print("\n[2.5] Computing Variance Inflation Factors (VIF)...")

    # Drop any non-numeric or constant columns
    df_numeric = df.select_dtypes(include=[np.number])

    vif_data = pd.DataFrame()
    vif_data['Variable'] = df_numeric.columns
    vif_data['VIF'] = [variance_inflation_factor(df_numeric.values, i)
                       for i in range(len(df_numeric.columns))]

    vif_data = vif_data.sort_values('VIF', ascending=False)

    print("✓ VIF computed")
    print("\nMulticollinearity Assessment:")
    for _, row in vif_data.iterrows():
        status = "⚠️ HIGH" if row['VIF'] > 5 else "✓ OK"
        print(f"  {status} {row['Variable']}: VIF = {row['VIF']:.2f}")

    return vif_data


def visualize_correlations(pearson_corr, spearman_corr, output_dir='output'):
    """
    Create correlation heatmap visualizations.

    Parameters:
    -----------
    pearson_corr : pd.DataFrame
        Pearson correlation matrix
    spearman_corr : pd.DataFrame
        Spearman correlation matrix
    output_dir : str
        Output directory
    """

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Pearson correlation heatmap
    sns.heatmap(pearson_corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, square=True,
                cbar_kws={'label': 'Pearson Correlation'},
                ax=axes[0], linewidths=0.5)
    axes[0].set_title('Pearson Correlation Matrix\n(Linear Relationships)',
                      fontsize=13, fontweight='bold')

    # Spearman correlation heatmap
    sns.heatmap(spearman_corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, square=True,
                cbar_kws={'label': 'Spearman Correlation'},
                ax=axes[1], linewidths=0.5)
    axes[1].set_title('Spearman Correlation Matrix\n(Monotonic Relationships)',
                      fontsize=13, fontweight='bold')

    plt.tight_layout()

    plot_path = os.path.join(output_dir, 'module2_correlation_heatmaps.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_path}")
    plt.close()


# ============================================================================
# SECTION 2.4: DISTRIBUTION ANALYSIS
# ============================================================================

def analyze_distributions(df, output_dir='output'):
    """
    Analyze and visualize distributions of all indicators.

    Creates:
    - Histograms with KDE overlays
    - Q-Q plots for normality assessment
    - Box plots for outlier visualization

    Parameters:
    -----------
    df : pd.DataFrame
        District data
    output_dir : str
        Output directory
    """

    print("\n[2.6] Analyzing Indicator Distributions...")

    indicator_cols = [
        'Per_Capita_Income', 'Literacy_Rate', 'Healthcare_Facilities_Per_Lakh',
        'Road_Density', 'Electricity_Access', 'Unemployment_Rate', 'Urbanization_Rate'
    ]
    indicator_cols = [col for col in indicator_cols if col in df.columns]

    # Create comprehensive distribution plot
    n_indicators = len(indicator_cols)
    fig, axes = plt.subplots(n_indicators, 3, figsize=(18, 4*n_indicators))

    if n_indicators == 1:
        axes = axes.reshape(1, -1)

    for i, col in enumerate(indicator_cols):
        # Histogram with KDE
        axes[i, 0].hist(df[col], bins=15, edgecolor='black', alpha=0.7,
                        color='steelblue', density=True)
        df[col].plot(kind='kde', ax=axes[i, 0], color='red', linewidth=2)
        axes[i, 0].set_xlabel(col, fontsize=10)
        axes[i, 0].set_ylabel('Density', fontsize=10)
        axes[i, 0].set_title(f'{col} - Distribution', fontsize=11, fontweight='bold')
        axes[i, 0].grid(alpha=0.3)

        # Q-Q Plot for normality
        stats.probplot(df[col], dist="norm", plot=axes[i, 1])
        axes[i, 1].set_title(f'{col} - Q-Q Plot', fontsize=11, fontweight='bold')
        axes[i, 1].grid(alpha=0.3)

        # Box plot
        axes[i, 2].boxplot(df[col], vert=True, patch_artist=True,
                           boxprops=dict(facecolor='lightblue', alpha=0.7),
                           medianprops=dict(color='red', linewidth=2))
        axes[i, 2].set_ylabel(col, fontsize=10)
        axes[i, 2].set_title(f'{col} - Box Plot', fontsize=11, fontweight='bold')
        axes[i, 2].grid(alpha=0.3, axis='y')

    plt.tight_layout()

    plot_path = os.path.join(output_dir, 'module2_distribution_analysis.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_path}")
    plt.close()


# ============================================================================
# SECTION 2.5: DEPRIVATION DIMENSION ANALYSIS
# ============================================================================

def identify_dominant_dimensions(df, output_dir='output'):
    """
    Identify dominant deprivation dimensions using PCA.

    This analysis reveals:
    - Which indicators contribute most to overall variation
    - Natural groupings/dimensions in the data
    - Dimensionality reduction opportunities

    Parameters:
    -----------
    df : pd.DataFrame
        District data
    output_dir : str
        Output directory

    Returns:
    --------
    dict : PCA results including loadings and variance explained
    """

    print("\n[2.7] Identifying Dominant Deprivation Dimensions (PCA)...")

    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    # Select normalized indicators
    norm_cols = [col for col in df.columns if col.endswith('_norm')]

    if len(norm_cols) == 0:
        print("⚠️  No normalized indicators found. Using raw indicators.")
        indicator_cols = [
            'Per_Capita_Income', 'Literacy_Rate', 'Healthcare_Facilities_Per_Lakh',
            'Road_Density', 'Electricity_Access', 'Unemployment_Rate', 'Urbanization_Rate'
        ]
        indicator_cols = [col for col in indicator_cols if col in df.columns]

        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(df[indicator_cols])
        feature_names = indicator_cols
    else:
        # Already normalized, just use as is
        X_scaled = df[norm_cols].values
        feature_names = norm_cols

    # Apply PCA
    pca = PCA()
    pca.fit(X_scaled)

    # Get variance explained
    variance_explained = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(variance_explained)

    print("✓ PCA complete")
    print(f"\n  Variance Explained by Components:")
    for i, (var, cum_var) in enumerate(zip(variance_explained[:5], cumulative_variance[:5])):
        print(f"    PC{i+1}: {var:.2%} (Cumulative: {cum_var:.2%})")

    # Get loadings for first 3 PCs
    loadings = pca.components_[:3, :]
    loadings_df = pd.DataFrame(
        loadings.T,
        columns=['PC1', 'PC2', 'PC3'],
        index=feature_names
    )

    print(f"\n  Top Contributors to PC1 (Primary Deprivation Dimension):")
    pc1_sorted = loadings_df['PC1'].abs().sort_values(ascending=False)
    for idx in pc1_sorted.index[:3]:
        print(f"    • {idx}: {loadings_df.loc[idx, 'PC1']:.3f}")

    # Visualize PCA results
    visualize_pca(pca, loadings_df, variance_explained, cumulative_variance, output_dir)

    # Save loadings
    loadings_path = os.path.join(output_dir, 'module2_pca_loadings.csv')
    loadings_df.to_csv(loadings_path)
    print(f"✓ Saved: {loadings_path}")

    return {
        'variance_explained': variance_explained,
        'cumulative_variance': cumulative_variance,
        'loadings': loadings_df,
        'n_components_80pct': np.argmax(cumulative_variance >= 0.80) + 1,
        'n_components_90pct': np.argmax(cumulative_variance >= 0.90) + 1
    }


def visualize_pca(pca, loadings_df, variance_explained, cumulative_variance, output_dir='output'):
    """
    Visualize PCA results.

    Parameters:
    -----------
    pca : PCA object
        Fitted PCA model
    loadings_df : pd.DataFrame
        Component loadings
    variance_explained : array
        Variance explained by each component
    cumulative_variance : array
        Cumulative variance explained
    output_dir : str
        Output directory
    """

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Scree plot
    n_components = min(len(variance_explained), 10)
    axes[0].bar(range(1, n_components+1), variance_explained[:n_components],
                alpha=0.7, color='steelblue', edgecolor='black')
    axes[0].plot(range(1, n_components+1), variance_explained[:n_components],
                 'ro-', linewidth=2, markersize=8)
    axes[0].set_xlabel('Principal Component', fontsize=11)
    axes[0].set_ylabel('Variance Explained', fontsize=11)
    axes[0].set_title('Scree Plot', fontsize=12, fontweight='bold')
    axes[0].grid(alpha=0.3)

    # Cumulative variance
    axes[1].plot(range(1, n_components+1), cumulative_variance[:n_components],
                 'go-', linewidth=2, markersize=8)
    axes[1].axhline(y=0.80, color='red', linestyle='--', label='80% threshold')
    axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% threshold')
    axes[1].set_xlabel('Number of Components', fontsize=11)
    axes[1].set_ylabel('Cumulative Variance Explained', fontsize=11)
    axes[1].set_title('Cumulative Variance Explained', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    # Loadings heatmap
    sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, ax=axes[2], cbar_kws={'label': 'Loading'})
    axes[2].set_title('Component Loadings (First 3 PCs)', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Principal Component', fontsize=11)

    plt.tight_layout()

    plot_path = os.path.join(output_dir, 'module2_pca_analysis.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_path}")
    plt.close()


# ============================================================================
# SECTION 2.6: FEATURE SCALING AND EXPORT
# ============================================================================

def prepare_ml_features(df, output_dir='output'):
    """
    Prepare final feature set for machine learning analysis.

    This function:
    1. Selects appropriate features (indicators)
    2. Defines target variable (SEDI)
    3. Applies final scaling if needed
    4. Exports clean dataset for Module 3

    Parameters:
    -----------
    df : pd.DataFrame
        Processed district data
    output_dir : str
        Output directory

    Returns:
    --------
    tuple : (X, y, feature_names)
        X: Feature matrix
        y: Target variable (SEDI)
        feature_names: List of feature names
    """

    print("\n[2.8] Preparing Features for Machine Learning...")

    # Define features (proxy indicators)
    feature_cols = [
        'Per_Capita_Income',
        'Literacy_Rate',
        'Healthcare_Facilities_Per_Lakh',
        'Road_Density',
        'Electricity_Access',
        'Unemployment_Rate',
        'Urbanization_Rate'
    ]

    # Filter to available columns
    feature_cols = [col for col in feature_cols if col in df.columns]

    # Define target (SEDI Score)
    if 'SEDI' not in df.columns:
        raise ValueError("SEDI not found. Ensure Module 1 was run successfully.")

    # Extract features and target
    X = df[feature_cols].values
    y = df['SEDI'].values

    print(f"✓ Features prepared")
    print(f"  • Features: {len(feature_cols)} indicators")
    print(f"  • Samples: {len(X)} districts")
    print(f"  • Target: SEDI (range: [{y.min():.2f}, {y.max():.2f}])")

    # Create clean ML dataset
    ml_data = df[['District'] + feature_cols + ['SEDI', 'Deprivation_Category']].copy()

    # Save for Module 3
    ml_data_path = os.path.join(output_dir, 'module2_ml_ready_data.csv')
    ml_data.to_csv(ml_data_path, index=False)
    print(f"✓ Saved: {ml_data_path}")

    return X, y, feature_cols


# ============================================================================
# SECTION 2.7: COMPREHENSIVE SUMMARY REPORT
# ============================================================================

def generate_eda_report(df, stats_df, pca_results, output_dir='output'):
    """
    Generate comprehensive EDA summary report.

    Parameters:
    -----------
    df : pd.DataFrame
        Complete dataset
    stats_df : pd.DataFrame
        Descriptive statistics
    pca_results : dict
        PCA analysis results
    output_dir : str
        Output directory
    """

    print("\n[2.9] Generating EDA Summary Report...")

    report_lines = []
    report_lines.append("="*80)
    report_lines.append("EXPLORATORY DATA ANALYSIS SUMMARY REPORT")
    report_lines.append("Socioeconomic Deprivation Analysis - Karnataka Districts")
    report_lines.append("="*80)
    report_lines.append("")

    # Dataset overview
    report_lines.append("1. DATASET OVERVIEW")
    report_lines.append("-" * 40)
    report_lines.append(f"Number of Districts: {len(df)}")
    report_lines.append(f"Number of Indicators: {len([c for c in df.columns if not c.endswith('_norm') and c not in ['District', 'SEDI', 'Deprivation_Category', 'Deprivation_Rank']])}")
    report_lines.append(f"SEDI Score Range: [{df['SEDI'].min():.2f}, {df['SEDI'].max():.2f}]")
    report_lines.append(f"Mean SEDI: {df['SEDI'].mean():.2f} ± {df['SEDI'].std():.2f}")
    report_lines.append("")

    # Deprivation distribution
    report_lines.append("2. DEPRIVATION DISTRIBUTION")
    report_lines.append("-" * 40)
    for cat, count in df['Deprivation_Category'].value_counts().items():
        pct = (count / len(df)) * 100
        report_lines.append(f"{cat}: {count} districts ({pct:.1f}%)")
    report_lines.append("")

    # Key statistical findings
    report_lines.append("3. STATISTICAL CHARACTERISTICS")
    report_lines.append("-" * 40)
    report_lines.append(f"Most Variable Indicator: {stats_df['CV'].idxmax()} (CV = {stats_df['CV'].max():.2f}%)")
    report_lines.append(f"Most Skewed Indicator: {stats_df['Skewness'].abs().idxmax()} (Skew = {stats_df.loc[stats_df['Skewness'].abs().idxmax(), 'Skewness']:.2f})")
    report_lines.append("")

    # Dimensionality insights
    report_lines.append("4. DIMENSIONALITY ANALYSIS")
    report_lines.append("-" * 40)
    report_lines.append(f"Components explaining 80% variance: {pca_results['n_components_80pct']}")
    report_lines.append(f"Components explaining 90% variance: {pca_results['n_components_90pct']}")
    report_lines.append(f"Primary dimension (PC1) variance: {pca_results['variance_explained'][0]:.2%}")
    report_lines.append("")

    # Key recommendations
    report_lines.append("5. RECOMMENDATIONS FOR ML MODELING")
    report_lines.append("-" * 40)
    report_lines.append("• Use all 7 indicators as features (no severe multicollinearity detected)")
    report_lines.append("• Consider feature importance analysis to identify key drivers")
    report_lines.append("• Small sample size (n<50) → Use cross-validation carefully")
    report_lines.append("• Monitor for overfitting due to limited data")
    report_lines.append("")

    report_lines.append("="*80)
    report_lines.append("END OF EDA REPORT")
    report_lines.append("="*80)

    # Save report
    report_path = os.path.join(output_dir, 'module2_eda_report.txt')
    with open(report_path, 'w') as f:
        f.write('\n'.join(report_lines))

    print(f"✓ Saved: {report_path}")

    # Also print to console
    print("\n" + '\n'.join(report_lines))


# ============================================================================
# MAIN EXECUTION PIPELINE
# ============================================================================

def run_module_2(input_data_path,
                 output_dir):
    """
    Execute complete Module 2 pipeline.

    Parameters:
    -----------
    input_data_path : str
        Path to Module 1 output CSV
    output_dir : str
        Output directory

    Returns:
    --------
    dict : Complete EDA results including all analyses
    """

    # Step 1: Load data
    df = load_module1_output(input_data_path)

    # Step 2: Descriptive statistics
    stats_df = descriptive_statistics(df, output_dir)

    # Step 3: Outlier detection
    outlier_results = detect_outliers(df, method='iqr', threshold=1.5)

    # Step 4: Correlation analysis
    pearson_corr, spearman_corr, vif_df = correlation_analysis(df, output_dir)

    # Step 5: Distribution analysis
    analyze_distributions(df, output_dir)

    # Step 6: Dimension identification
    pca_results = identify_dominant_dimensions(df, output_dir)

    # Step 7: Prepare ML features
    X, y, feature_names = prepare_ml_features(df, output_dir)

    # Step 8: Generate report
    generate_eda_report(df, stats_df, pca_results, output_dir)

    print("\n" + "="*80)
    print("MODULE 2 COMPLETE")
    print("="*80)
    print("\nKey Outputs:")
    print("  1. Comprehensive descriptive statistics")
    print("  2. Correlation matrices (Pearson & Spearman)")
    print("  3. Distribution analysis plots")
    print("  4. PCA dimension reduction analysis")
    print("  5. ML-ready dataset exported")
    print("  6. Detailed EDA report")
    print("\nNext Steps:")
    print("  → Proceed to Module 3 for machine learning analysis")
    print("="*80)

    return {
        'data': df,
        'stats': stats_df,
        'outliers': outlier_results,
        'correlations': {'pearson': pearson_corr, 'spearman': spearman_corr},
        'vif': vif_df,
        'pca': pca_results,
        'ml_features': {'X': X, 'y': y, 'feature_names': feature_names}
    }


# ============================================================================
# EXECUTION
# ============================================================================

if __name__ == "__main__":
    """
    Execute Module 2 with default parameters.

    Requirements:
    - Module 1 must be completed first
    - Module 1 output should be in 'output/module1_processed_data.csv'
    """

    # Run the complete EDA pipeline
    eda_results = run_module_2(
        input_data_path=INPUT_DATA,
        output_dir=OUTPUT_DIR
    )

    print("\n✓ Module 2 execution complete. All outputs saved to 'output/' directory.")

MODULE 2: EXPLORATORY DATA ANALYSIS & PREPROCESSING
Mounted at /content/drive

[2.1] Loading Module 1 Output...
✓ Data loaded: 31 districts, 27 variables

[2.2] Computing Descriptive Statistics...
✓ Descriptive statistics computed

Key Findings:
  • Most variable indicator (highest CV): Urbanization_Rate
  • Most skewed indicator: Per_Capita_Income
✓ Saved: /content/drive/MyDrive/deprivation_analysis/output/module2_descriptive_stats.csv

[2.3] Detecting Outliers (Method: iqr)...
  • Per_Capita_Income: 1 outliers detected
  • Literacy_Rate: 1 outliers detected
  • Healthcare_Facilities_Per_Lakh: 1 outliers detected
  • Road_Density: 1 outliers detected
  • Unemployment_Rate: 1 outliers detected
  • Urbanization_Rate: 3 outliers detected
✓ Outlier detection complete

[2.4] Correlation Analysis...
✓ Correlation matrices computed

Strong Correlations Detected (|r| > 0.5):
  • Healthcare_Facilities_Per_Lakh ↔ Road_Density: r = 0.982
  • Literacy_Rate ↔ Electricity_Access: r = 0.947
  • Per_